# behaviors

> The client runtime: delegated event handlers that keep working across htmx swaps

In [ ]:
#| default_exp behaviors

The registry stores a component's behaviors as records: a selector, an event, a capture flag and a handler body. Here we turn those records into code that runs in the browser. Attaching a listener to each element as it appears works for a static page. In a FastHTML app, where htmx keeps swapping in new elements, per-element listeners would have to be attached again after every swap.

Instead we delegate: one listener on the document handles each event. When the event fires anywhere on the page, the listener checks whether the target is inside an element that matches a rule's selector. Because the listener checks the live DOM each time the event fires, elements swapped in later behave like those in the original page, with no per-element state and no mutation observer. A hyperelement redefined in a live notebook takes effect at the next page render, because the rules are plain data generated again on each render.

In [ ]:
from fastcore.utils import *
from answerui.registry import Registry

In [ ]:
from answerui.registry import reg, hypelt
from fastcore.test import *

## The runtime

The script below is the whole runtime. The runtime receives the rules as a list of records, each with a selector `sel`, the event names `evs` to listen for, a capture flag `cap`, an `init` flag and a handler function `fn`. The runtime groups them by event and phase, and installs one listener on the document for each pair, however many rules share it. When an event fires, the listener finds the nearest element matching each rule's selector, starting at the target and walking up its ancestors. If it finds a match, it passes that element to the rule's handler.

An `init` rule's handler runs when a matching element arrives in the document. The runtime calls these handlers from `htmx.onLoad`, htmx's hook for arriving content. The hook calls back with the root of newly swapped content, or with the body for the initial page. The runtime then calls each `init` handler on every element in that content that matches the rule's selector, the root included. Walking up from the root instead would run a handler for an ancestor that was already on the page, as though it had just arrived. The handler's `e` is an `Event` of type `init`, created for the call and never dispatched. htmx must load before this script, because the script calls `htmx.onLoad`. FastHTML's default headers and `demo`, the notebook preview, both include htmx's script before this one.

The script runs only once per document: FastHTML sends the headers only with a full page, never with a fragment returned to an htmx request, and htmx discards the `<head>` of any full page it swaps.

Handlers are async functions, so a failure in one rejects its own promise and leaves the other rules on that event running. Listening in the capture phase is opt-in, for events like `toggle` and `scroll` that don't bubble. A regenerated script takes effect only in a fresh page or preview: a second copy injected into the same document would add its listeners to the first copy's.

In [ ]:
_runtime = r"""(() => {
%s
const rules = [%s];
const by = {};
rules.forEach(r => r.evs.forEach(ev => (by[ev + '|' + (r.cap ? 1 : 0)] ??= []).push(r)));
for (const k in by) {
  const [ev, cap] = k.split('|');
  document.addEventListener(ev, e => {
    const t = e.target;
    if (!(t instanceof Element)) return;
    for (const r of by[k]) {
      const el = t.closest(r.sel);
      if (el) r.fn(el, e);
    }
  }, {capture: cap === '1'});
}
const init = rules.filter(r => r.init);
if (init.length) htmx.onLoad(root => {
  const e = new Event('init');
  for (const r of init) {
    if (root.matches(r.sel)) r.fn(root, e);
    root.querySelectorAll(r.sel).forEach(el => r.fn(el, e));
  }
});
})();"""

## Entries

Each behavior record becomes one object literal in the runtime's `rules` list. Two event names belong to the library: `init`, when an element arrives in the document, and `cleanup`, before htmx removes one. Any other name is a DOM event. A leading `:` stands for `htmx:`, as in `':after-request'` for `htmx:after-request`.

htmx spells its events three ways. htmx 2 dispatches `htmx:afterSwap` and the alias `htmx:after-swap`. htmx 4 renamed its events to the form `htmx:after:swap`. FastHTML's `htmx4=True` sets `metaCharacter` to `-`, which htmx 4 puts in place of every colon, giving `htmx-after-swap`. `_event` converts every spelling of an htmx event to one canonical name: `htmx:` plus the hyphenated form. `_spellings` lists the DOM event names for that canonical name. The runtime listens for all these names, but only one fires on a given page. The library's own two event names, `init` and `cleanup`, are handled separately. `init` is not a DOM event: the runtime calls `init` handlers from `htmx.onLoad`, which listens for the arrival event of whichever htmx version is loaded. `cleanup` has different names in the two versions: `htmx:beforeCleanupElement` in htmx 2 and `htmx:before:cleanup` in htmx 4. `_spellings` returns them from a fixed list, `_cleanup`. Under htmx 4, `cleanup` fires only for elements htmx initialised: those carrying `hx-` attributes. Each distinct handler body becomes one async function taking `el` and `e`. These functions are declared before the list, and each entry refers to its function:

In [ ]:
_cleanup = ['htmx:before-cleanup-element', 'htmx:before:cleanup', 'htmx-before-cleanup']

def _event(ev):
    "The canonical event name: a leading `:` stands for `htmx:`, and every spelling of an htmx event becomes `htmx:` plus its hyphenated name"
    if ev.startswith(':'): ev = 'htmx'+ev
    if not ev.startswith(('htmx:', 'htmx-')): return ev
    return 'htmx:' + re.sub(r'(?<=[a-z])(?=[A-Z])', '-', ev[5:]).lower().replace(':', '-')

def _spellings(ev):
    "The DOM event names that the runtime listens for to handle canonical event `ev`. For an htmx event, these are htmx 2's hyphenated alias, htmx 4's colon form, and htmx 4's form with every colon replaced by FastHTML's `metaCharacter`"
    if ev == 'init': return []
    if ev == 'cleanup': return _cleanup
    if not ev.startswith('htmx:'): return [ev]
    name = ev[5:]
    return uniqueify([ev, 'htmx:'+name.replace('-', ':'), 'htmx-'+name])

def _entry(r, fn):
    ev = _event(r['ev'])
    return f'{{sel: {dumps(r["sel"])}, evs: {dumps(_spellings(ev))}, cap: {dumps(r["cap"])}, init: {dumps(ev=="init")}, fn: {fn}}}'


`_event` converts every spelling of an htmx event to the same canonical name, and `_spellings` gives the DOM names the runtime listens for:

In [ ]:
for ev in [':after-swap', 'htmx:afterSwap', 'htmx:after:swap', 'htmx-after-swap']: test_eq(_event(ev), 'htmx:after-swap')
_spellings('htmx:after-swap')

['htmx:after-swap', 'htmx:after:swap', 'htmx-after-swap']

## Building the script

A behavior's identity is its selector, its event, and its capture flag, with the event name made canonical first. Declaring the same identity again replaces the earlier handler. Only the last declaration in registry order is included in the script. Re-registering a rule keeps its position in registry order, so re-running a component's definition never replaces an application's later override. A replacement handler also keeps the original handler's position, so handlers for an event run in the order their identities were first declared. Because the registry compares selector strings and does not parse CSS, two different selectors that pick the same elements are two behaviors, and both handlers run.

`behaviors` returns the effective handler for each identity, with the key of the rule that declared it. `build_js` puts these records into the runtime script:


In [ ]:
@patch
def behaviors(self:Registry):
    "The effective behavior per (selector, event, capture): the last declared in registry order, with the key of the rule that declared it"
    res = {}
    for k,r in self.rules.items():
        for b in r.on: res[(b['sel'], _event(b['ev']), b['cap'])] = dict(b, ev=_event(b['ev']), rule=k)
    return res

@patch
def build_js(self:Registry):
    "The behavior script: one function per distinct handler, one entry per effective behavior"
    bs = self.behaviors().values()
    fns = {js: f'f{i}' for i,js in enumerate(uniqueify([b['js'] for b in bs]))}
    decls = '\n'.join(f'async function {nm}(el, e) {{ {js} }}' for js,nm in fns.items())
    return _runtime % (decls, ',\n'.join(_entry(b, fns[b['js']]) for b in bs))


As examples, let's make a notice that removes itself after a delay or when a child marked `dismiss` is clicked, and a menu that focuses its first item when its popover opens. The notice's delayed dismissal uses an `init` rule. htmx fires the arrival event for the initial page and for every swapped fragment, but never for content already on the page, so swapping something inside a notice doesn't restart its timer. Since `toggle` doesn't bubble, the menu rule uses the capture phase. Because the capture listener also receives `toggle` events from elements inside the menu, such as a `details` element, the handler checks that `e.target` is the menu itself:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
reg.rule('dropdown', sel='[popover][dropdown]', on={('&', 'toggle', True): r"if (e.target === el && e.newState === 'open') el.querySelector('a, button')?.focus()"})
print('\n'.join(l for l in reg.build_js().splitlines() if 'notice-box' in l or '[dropdown]' in l))

async function f1(el, e) { el.closest('notice-box').remove() }
const rules = [{sel: ":is(notice-box)[autodismiss]", evs: [], cap: false, init: true, fn: f0},
{sel: ":is(notice-box) [dismiss]", evs: ["click"], cap: false, init: false, fn: f1},
{sel: ":is([popover][dropdown])", evs: ["toggle"], cap: true, init: false, fn: f2}];


The `init` entry has `init: true` and `evs: []`, because the runtime calls its handler from `htmx.onLoad`. The `toggle` entry has `cap: true`, and the click entry has `cap: false` and `init: false`. The runtime groups the rules into listeners in the browser, so `build_js` generates only the functions and the rule list.

In [ ]:
s = reg.build_js()
assert 'init: true' in s and 'htmx.onLoad' in s and 'cap: true' in s

A rule often gives one handler to several events. Here a form is marked `dirty` on `input` and again on `change`, since a select fires only the second. The script declares that handler body once, and both entries refer to the same function:

In [ ]:
_dirty = r"el.toggleAttribute('dirty', [...el.elements].some(c => c.value !== c.defaultValue))"
reg.rule('dirty', sel='form', on={('&', 'input'): _dirty, ('&', 'change'): _dirty})
s = reg.build_js()
test_eq(s.count(_dirty), 1)
test_eq(s.count('":is(form)"'), 2)

An application that wants confirmation before a notice is dismissed registers the same selector and event through the component. `&` expands to `:is(notice-box)`, as it did in the component's definition, so the behavior's identity matches and the handler that asks for confirmation replaces the original. The `rule` field of each effective behavior names the rule that declared its handler:

In [ ]:
NoticeBox.rule('confirm-dismiss', on={('& [dismiss]', 'click'): r"if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()"})
{k: b['rule'] for k,b in reg.behaviors().items()}

{(':is(notice-box)[autodismiss]', 'init', False): 'NoticeBox.base',
 (':is(notice-box) [dismiss]', 'click', False): 'NoticeBox.confirm-dismiss',
 (':is([popover][dropdown])', 'toggle', True): 'dropdown',
 (':is(form)', 'input', False): 'dirty',
 (':is(form)', 'change', False): 'dirty'}

The script has one click handler for the dismiss button: the one that asks for confirmation. `NoticeBox.behaviors` still lists both declarations, because it reports what was declared on the component; `reg.behaviors` reports what goes into the script:

In [ ]:
s = reg.build_js()
test_eq(s.count('":is(notice-box) [dismiss]"'), 1)
assert 'confirm(' in s
NoticeBox.behaviors

[{'sel': ':is(notice-box)[autodismiss]',
  'ev': 'init',
  'js': "await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "el.closest('notice-box').remove()",
  'cap': False},
 {'sel': ':is(notice-box) [dismiss]',
  'ev': 'click',
  'js': "if (confirm('Dismiss this notice?')) el.closest('notice-box').remove()",
  'cap': False}]

Re-registering the component's definition does not displace the application's override:

In [ ]:
NoticeBox = hypelt('NoticeBox', layer='components', doc='A notice that dismisses itself.', on={
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss'))); el.remove()",
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()"})
test_eq(reg.behaviors()[(':is(notice-box) [dismiss]', 'click', False)]['rule'], 'NoticeBox.confirm-dismiss')
[k for k in reg.behaviors() if k[1]=='init']

[(':is(notice-box)[autodismiss]', 'init', False)]

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()